# 01 - Data Exploration

Load one ticker, run it through the full Section A pipeline (clean -> features -> behavioral -> label) and eyeball the result.

In [ ]:
import sys, os
sys.path.append(os.path.abspath('..'))  # so 'import src...' works from notebooks/

import pandas as pd
import matplotlib.pyplot as plt

from src.config import load_config
config = load_config('../config.yaml')
ticker = config['data']['tickers'][0]
config['data']['raw_dir'] = '../data/raw'  # paths are relative to project root
ticker

## Load and clean

In [ ]:
from src.data.loader import load_ohlcv
from src.data.cleaner import clean_ohlcv

raw = load_ohlcv(ticker, config['data']['start_date'], config['data']['end_date'],
                 raw_dir=config['data']['raw_dir'])
df = clean_ohlcv(raw)
print(df.shape)
df.head()

In [ ]:
df['Close'].plot(figsize=(11, 4), title=f'{ticker} adjusted close')
plt.show()

## Quantitative features

In [ ]:
from src.features import build_features
df = build_features(df, config)
df.filter(regex='return|sma|ema|rsi|momentum|vol').head()

## Behavioral features

In [ ]:
from src.behavioral import build_behavioral_features
df = build_behavioral_features(df, config)
df[['fear', 'greed', 'herd', 'regime']].dropna().head()

In [ ]:
df[['fear', 'greed', 'herd']].plot(figsize=(11, 4), title='Behavioral indicators (normalized)')
plt.show()

## Label and final matrix

In [ ]:
from src.data.labels import make_direction_label
df['target'] = make_direction_label(df, config['labels']['horizon'], config['labels']['threshold'])

model_df = df.dropna()
print('rows after dropping warmup + final NaNs:', len(model_df))
print('class balance:')
model_df['target'].value_counts(normalize=True)